# Mission Analysis (preliminary)
### Roadrunner propulsion reverse-engineering study · notebook 6 of 13

**Purpose.** Fly the sample mission (ADR-0018) with the thrust the constraint analysis demands:
weight fraction and fuel per segment, total fuel, and what that leaves for the empty vehicle.
It answers:

> *How much fuel does the mission burn, and what take-off mass is consistent with an engine
> pair of the size the constraints require?*

**First iteration, rough by intent (ADR-0029).** Constant TSFC from the catalogue median, the
algebraic turbojet lapse, the hand-estimate polar. Climb and acceleration use the
energy-height method; level flight and loiter integrate $dm = C\,D\,dt$; launch and landing
are hover at $T=W$. The cycle analysis closes the numbers later.

This is the *preliminary* analysis (ADR-0031). It is repeated in notebook 10 with the off-design
cycle results; compare the two.

---
## 0 · Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # bare-checkout fallback

from roadrunner_re import analysis_outputs as ao
from roadrunner_re import reports
from roadrunner_re.analysis_inputs import load_analysis_inputs
from roadrunner_re.design_point import solve_design_point
from roadrunner_re.notebook import nb_setup
from roadrunner_re.plots import analysis as plots

paths = nb_setup()
inputs = load_analysis_inputs()
design = solve_design_point(inputs)          # the single call site (ADR-0029)
sweep = ao.takeoff_mass_sweep(inputs, ao.sweep_masses(inputs))


---
## 1 · Weight fractions

Each segment's mass at its start, its weight fraction and the fuel it burns. The weight at the
start of a segment is the $\beta$ the constraint analysis used, so the two notebooks describe one
solution (converged in the number of iterations printed below).

In [ ]:
print(reports.weight_fraction_table(design))
print(f"\nconverged in {design.iterations} iterations")

In [ ]:
fig = plots.mission_weights_figure(design, paths.figures / "mission_weights.png")

The descent is an idle glide timed by the mission's sink rate; its fuel is an assumed weight
fraction (`config/analysis.yaml`), because idle SFC is not modelled. The transition onto the
engines before landing is flown like hover, thrust equal to weight (ADR-0038).

---
## 2 · What the take-off mass leaves for the vehicle

Take-off mass is not solved for here. A structural mass build-up (ADR-0008) is not built, so the
*implied empty mass* — take-off mass minus payload minus fuel with reserve — is reported next to
the mass of the two engines from the catalogue power law, rather than closed.

In [ ]:
m = design.mission
print(f"take-off mass          {m.takeoff_mass_kg:7.2f} kg")
print(f"payload                {m.payload_mass_kg:7.2f} kg")
print(f"fuel incl. reserve     {m.total_fuel_kg:7.2f} kg")
print(f"implied empty mass     {m.implied_empty_mass_kg:7.2f} kg")
print(f"thrust, pair installed {design.thrust_SL_N:7.0f} N  ({design.thrust_per_engine_N:.0f} N per engine)")
print(f"propulsion (2 engines) {design.propulsion_mass_kg:7.2f} kg = {design.propulsion_fraction_of_empty:.0%} of implied empty mass")

In [ ]:
fig = plots.mass_sweep_figure(sweep, paths.figures / "mass_sweep.png")

---
## 3 · Handoff

In [ ]:
handoff = ao.write_handoff(ao.mission_out(inputs, design, sweep), paths.out / "mission_preliminary.yaml")
print(f"wrote {handoff.relative_to(paths.repo)}")

---
## Findings

- **The loiter burns about half the mission fuel** at constant TSFC, and is the segment most
  understated by the model: a real turbojet at part throttle is thirstier.
- **Dash and return are small**; climb is cheap because the energy-height method makes its
  fuel nearly independent of the thrust margin.
- **Fuel is a modest share of take-off mass**, and the propulsion mass a modest share of the
  implied empty mass; neither closes the vehicle, because no structure build-up exists yet.
- **Standing finding:** every number is first-iteration. TSFC is constant, the glide's fuel is an
  assumed fraction, the launch-side transition is not flown, the landing-side one is flown at
  thrust equal to weight (an upper estimate) and installation losses are not applied (see the
  `simplifications` list in `out/mission_preliminary.yaml`).